# Upload the distilled LFM checkpoint to Hugging Face

This notebook takes the completed checkpoint from Google Drive, creates a clean inference-only staging directory, installs a ShellAI-compatible chat template, tests the checkpoint locally, uploads it to a Hugging Face model repository, and reloads the uploaded model in Q4/NF4 for a round-trip test.

Start with a fresh GPU runtime. The upload defaults to a **private** repository. Create a Colab secret named `HF_TOKEN` containing a Hugging Face token with write access. Optionally create `HF_REPO_ID` with a value such as `username/lfm2-nl2bash-distilled`.

In [ ]:
!nvidia-smi
# Text-only model: remove optional packages that commonly conflict in Colab.
!pip -q uninstall -y peft torchvision
!pip -q install --upgrade "transformers>=5.0,<6" "huggingface-hub>=0.34,<2" "safetensors>=0.6,<1" "bitsandbytes>=0.47,<1" "accelerate>=1.10,<2" "jinja2>=3.1,<4"
!python -c "import numpy, torch, transformers; print('numpy', numpy.__version__, '| torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__)"

In [ ]:
from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import HfApi

drive.mount('/content/drive')

CHECKPOINT_ROOT = Path('/content/drive/MyDrive/shellai/lfm2-nl2bash-distilled')
PRIVATE_REPO = True

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception as error:
    raise RuntimeError('Add a write-capable HF_TOKEN in Colab Secrets.') from error
if not hf_token:
    raise RuntimeError('HF_TOKEN is empty; add it in Colab Secrets.')

api = HfApi(token=hf_token)
account = api.whoami()
try:
    secret_repo_id = userdata.get('HF_REPO_ID')
except Exception:
    secret_repo_id = None
HF_REPO_ID = secret_repo_id or f"{account['name']}/lfm2-nl2bash-distilled"

candidates = [CHECKPOINT_ROOT / 'final', CHECKPOINT_ROOT / 'checkpoint-last']
MODEL_DIR = next(
    (path for path in candidates if (path / 'config.json').is_file()
     and any(path.glob('model*.safetensors'))),
    None,
)
if MODEL_DIR is None:
    raise FileNotFoundError(
        f'No model checkpoint found under {CHECKPOINT_ROOT}; checked {candidates}'
    )
print('Hugging Face account:', account['name'])
print('Repository:', HF_REPO_ID, '(private)' if PRIVATE_REPO else '(public)')
print('Checkpoint:', MODEL_DIR)

## Build a clean, ShellAI-compatible model directory

Only inference artifacts are copied. `training_state.pt` and optimizer state remain on Drive. The replacement chat template deliberately normalizes ShellAI's existing messages to the exact system prompt, user prefix, and empty-thinking prefix used during distillation.

In [ ]:
import json, shutil
from transformers import AutoConfig, AutoTokenizer, GenerationConfig

STAGING_DIR = Path('/content/lfm2-nl2bash-hub-upload')
if STAGING_DIR.exists():
    shutil.rmtree(STAGING_DIR)
STAGING_DIR.mkdir(parents=True)

weight_files = sorted(MODEL_DIR.glob('model*.safetensors'))
index_files = sorted(MODEL_DIR.glob('model*.safetensors.index.json'))
if not weight_files:
    raise FileNotFoundError(f'No safetensors model files in {MODEL_DIR}')
for source in weight_files + index_files:
    shutil.copy2(source, STAGING_DIR / source.name)

for metadata_name in ('distillation_config.json', 'vocabulary_transplant.json'):
    source = MODEL_DIR / metadata_name
    if not source.exists():
        source = CHECKPOINT_ROOT / metadata_name
    if source.exists():
        shutil.copy2(source, STAGING_DIR / metadata_name)

config = AutoConfig.from_pretrained(MODEL_DIR, local_files_only=True)
config.use_cache = True
config.save_pretrained(STAGING_DIR)

TRAINING_SYSTEM = (
    'You translate natural-language requests into exactly one Bash command. '
    'Return only the command, with no Markdown, explanation, or alternatives.'
)
SHELLAI_CHAT_TEMPLATE = '''{%- set shellai_system = "You translate natural-language requests into exactly one Bash command. Return only the command, with no Markdown, explanation, or alternatives." -%}
{{- bos_token }}
{{- "<|im_start|>system\n" + shellai_system + "<|im_end|>\n" }}
{%- for message in messages %}
{%- if message["role"] == "user" %}
{{- "<|im_start|>user\n" + (message["content"] | replace("Generate single Bash command: ", "Generate one Bash command: ")) + "<|im_end|>\n" }}
{%- elif message["role"] == "assistant" %}
{{- "<|im_start|>assistant\n" + message["content"] + "<|im_end|>\n" }}
{%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
{{- "<|im_start|>assistant\n<think></think>\n" }}
{%- endif %}'''

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
tokenizer.chat_template = SHELLAI_CHAT_TEMPLATE
tokenizer.save_pretrained(STAGING_DIR)

try:
    generation_config = GenerationConfig.from_pretrained(MODEL_DIR, local_files_only=True)
except OSError:
    generation_config = GenerationConfig.from_model_config(config)
generation_config.do_sample = False
generation_config.eos_token_id = tokenizer.eos_token_id
generation_config.pad_token_id = tokenizer.pad_token_id
generation_config.save_pretrained(STAGING_DIR)

shellai_messages = [
    {'role': 'system', 'content': 'You are a helpful assistant that translates natural language to bash commands.'},
    {'role': 'user', 'content': 'Generate single Bash command: list files'},
]
rendered = tokenizer.apply_chat_template(
    shellai_messages, tokenize=False, add_generation_prompt=True
)
expected = (
    '<|startoftext|><|im_start|>system\n' + TRAINING_SYSTEM + '<|im_end|>\n'
    '<|im_start|>user\nGenerate one Bash command: list files<|im_end|>\n'
    '<|im_start|>assistant\n<think></think>\n'
)
assert rendered == expected, 'ShellAI template does not match the training prompt'
assert config.vocab_size == 128000
assert max(tokenizer.get_vocab().values()) < config.vocab_size

model_card = f'''---
library_name: transformers
pipeline_tag: text-generation
base_model:
- LiquidAI/LFM2-350M
- LiquidAI/LFM2.5-2.6B
license: other
license_name: lfm1.0
license_link: https://huggingface.co/LiquidAI/LFM2-350M/blob/main/LICENSE
tags:
- lfm2
- knowledge-distillation
- nl2bash
- shellai
---

# LFM2 NL2Bash distilled for ShellAI

This model uses the LFM2-350M transformer body with a vocabulary transplanted to the LFM2.5-2.6B tokenizer, then full-weight response-only distillation on NL2Bash. The expanded untied input/output matrices make it approximately 482M parameters.

The included chat template adapts ShellAI's current message format to the exact prompt used during training.

```bash
python -m pip install -U 'transformers>=5,<6'
hf auth login
ai, --model {HF_REPO_ID} list all files modified today
```

The repository stores portable full-precision weights. It can be loaded with bitsandbytes NF4/Q4 on CUDA; the current ShellAI CLI itself does not yet request Q4 loading.
'''
(STAGING_DIR / 'README.md').write_text(model_card, encoding='utf-8')

print('Staged files:')
for path in sorted(STAGING_DIR.iterdir()):
    print(f'  {path.name:36} {path.stat().st_size / (1024**2):9.2f} MiB')
print('\nShellAI-rendered prompt matches the training prompt exactly.')

## Validate the staged checkpoint before uploading

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime for checkpoint validation.')
model = AutoModelForCausalLM.from_pretrained(
    STAGING_DIR, dtype=torch.bfloat16, device_map={'': 0}, local_files_only=True
)
model.eval()
assert model.get_input_embeddings().weight.shape[0] == config.vocab_size
assert model.get_output_embeddings().weight.shape[0] == config.vocab_size

inputs = tokenizer.apply_chat_template(
    shellai_messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors='pt'
).to('cuda')
with torch.inference_mode():
    generated = model.generate(
        **inputs, max_new_tokens=64, do_sample=False,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
    )
continuation = generated[0, inputs['input_ids'].shape[1]:]
local_prediction = tokenizer.decode(continuation, skip_special_tokens=True).strip()
if not local_prediction:
    raise RuntimeError('Local checkpoint generated an empty response')
print('Parameters:', f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')
print('Local prediction:', local_prediction)
del model, inputs, generated
gc.collect(); torch.cuda.empty_cache()

## Upload the clean model repository

This creates the repository if necessary and uploads the staged folder using Hugging Face's resumable/chunked upload path. Rerunning the cell is safe and creates another commit rather than duplicating model storage.

In [ ]:
repo_url = api.create_repo(
    repo_id=HF_REPO_ID, repo_type='model', private=PRIVATE_REPO, exist_ok=True
)
commit = api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type='model',
    folder_path=STAGING_DIR,
    commit_message='Upload distilled LFM2 NL2Bash checkpoint for ShellAI',
)
remote_files = set(api.list_repo_files(HF_REPO_ID, repo_type='model'))
required_remote = {'config.json', 'tokenizer.json', 'chat_template.jinja', 'README.md'}
missing = required_remote - remote_files
if missing or not any(name.endswith('.safetensors') for name in remote_files):
    raise RuntimeError(f'Hub upload is incomplete; missing={sorted(missing)}')
print('Repository:', repo_url)
print('Commit:', commit)
print('Uploaded files:', len(remote_files))

## Reload from Hugging Face in Q4 and test the ShellAI contract

This deliberately reloads from the Hub rather than Drive, proving that the uploaded repository is independently usable.

In [ ]:
from transformers import BitsAndBytesConfig

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
hub_tokenizer = AutoTokenizer.from_pretrained(HF_REPO_ID, token=hf_token)
hub_model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_ID, token=hf_token, quantization_config=quantization,
    device_map={'': 0}, dtype=torch.bfloat16,
)
hub_inputs = hub_tokenizer.apply_chat_template(
    shellai_messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors='pt'
).to('cuda')
with torch.inference_mode():
    hub_generated = hub_model.generate(
        **hub_inputs, max_new_tokens=64, do_sample=False,
        eos_token_id=hub_tokenizer.eos_token_id,
        pad_token_id=hub_tokenizer.pad_token_id,
    )
hub_continuation = hub_generated[0, hub_inputs['input_ids'].shape[1]:]
hub_prediction = hub_tokenizer.decode(hub_continuation, skip_special_tokens=True).strip()
if not hub_prediction:
    raise RuntimeError('Uploaded Q4 model generated an empty response')
print('Uploaded Q4 prediction:', hub_prediction)
print(f'\nShellAI command:\nai, --model {HF_REPO_ID} list files')

## Test from ShellAI

On the machine where ShellAI is installed:

```bash
python -m pip install -U 'transformers>=5,<6'
hf auth login
ai, --model YOUR_USERNAME/lfm2-nl2bash-distilled list all files modified today
```

Replace the repository name if you changed `HF_REPO_ID`. Private repositories require `hf auth login`. The current ShellAI implementation loads standard Transformers weights, so this command tests the model integration but does not itself enable Q4. The preceding Colab cell verifies that the same uploaded repository supports NF4/Q4 loading on CUDA.